# Module 17: Data Wrangling with Pandas

**Estimated time: 90-120 minutes**

---

## Learning Objectives

- Handle missing values and duplicates in biological datasets
- Convert data types and clean messy annotation tables
- Reshape data between wide and long formats (melt, pivot, stack/unstack)
- Use string operations to extract information from annotation columns
- Apply custom transformations with `apply`, `transform`, and `pipe`
- Merge gene annotations with expression data

### Why Data Wrangling?

Real biological data is messy. Gene names have inconsistent capitalization, clinical tables have missing values, expression matrices arrive in the wrong shape, and annotation files mix numeric IDs with free text. Before any analysis, you must clean and reshape the data. This module teaches the Pandas tools for that.

```
Raw Data        Clean          Reshape         Analyze
 Missing   -->  Fill/Drop  -->  Melt/Pivot -->  Ready for
 Dupes          Fix types       Merge           statistics
 Messy          Strings         Stack
```

---

## Why data wrangling matters in bioinformatics

Real biological data is messy: RNA-seq count tables have missing samples, clinical metadata has inconsistent capitalization and mixed types, annotation tables have duplicate gene entries, and data is almost always in the wrong shape (wide vs long) for the tool you want to run. Pandas wrangling is the unavoidable first step before any analysis.

In [ ]:
## How to use this notebook

Run cells from top to bottom on the first pass — later cells depend on functions and variables defined earlier. Once you have run through the notebook, feel free to modify parameters and re-run individual sections.

Each section has runnable examples first, followed by exercises. Try the exercise before looking at the solution cell below it.

## Complicated moments explained

**1. `fillna` is not in-place by default**
`df['col'].fillna(0)` returns a new Series — it does not modify `df`. Use `df['col'] = df['col'].fillna(0)` or `df.fillna({'col': 0}, inplace=True)`.

**2. `melt` vs `stack`**
`melt` converts specific value columns to rows (wide → long), keeping id columns unchanged. `stack` pivots the innermost column level into the row index. For typical expression data reshaping, `melt` is what you want.

**3. `groupby` and the index**
By default, `groupby('gene_id').agg(...)` moves `gene_id` into the index. Use `as_index=False` to keep it as a regular column, avoiding surprises when merging downstream.

**4. String operations on columns with NaN**
`df['col'].str.upper()` silently returns `NaN` for missing values instead of raising an error. Always check for unexpected NaNs after string operations on annotation columns.

In [ ]:
# A messy gene expression dataset with missing values and duplicates
messy_expr = pd.DataFrame({
    'gene_id':  ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'TP53', 'PIK3CA'],
    'sample_1': [5.2,    np.nan,   7.1,    2.4,   4.5,    5.3,    6.3],
    'sample_2': [5.4,    3.8,      7.3,    np.nan, 4.2,   5.1,    6.1],
    'sample_3': [5.0,    3.5,      6.9,    2.6,   4.8,    5.5,    np.nan],
    'gene_type': ['TSG', 'TSG', 'Oncogene', 'Oncogene', 'Oncogene', 'TSG', 'Oncogene'],
})

print("Messy expression data:")
print(messy_expr)
print(f"\nMissing values per column:\n{messy_expr.isna().sum()}")
print(f"\nTotal missing values: {messy_expr.isna().sum().sum()}")

In [ ]:
# Strategy 1: Drop rows with any missing value
dropped = messy_expr.dropna()
print(f"After dropna(): {len(dropped)} rows (from {len(messy_expr)})")
print(dropped)

In [ ]:
# Strategy 2: Fill with column mean (common for expression data)
sample_cols = ['sample_1', 'sample_2', 'sample_3']
filled = messy_expr.copy()
filled[sample_cols] = filled[sample_cols].fillna(filled[sample_cols].mean())

print("After filling NaN with column means:")
print(filled)

In [ ]:
# Strategy 3: Fill with row mean (gene-wise imputation)
filled_row = messy_expr.copy()
row_means = filled_row[sample_cols].mean(axis=1)
for col in sample_cols:
    filled_row[col] = filled_row[col].fillna(row_means)

print("After filling NaN with row means:")
print(filled_row)

### Biological context: when to drop vs. fill

- **Drop** when the fraction of missing data is small and the missingness is random (e.g., a few failed library preps). Dropping preserves unbiased statistics.
- **Fill with mean/median** when you need a complete matrix for downstream analysis (e.g., clustering, PCA) and missingness is low.
- **Fill with 0** for count data when NaN genuinely means "no reads detected."
- Never silently fill without documenting which strategy you used.

---

## Part 2: Handling Duplicates

In [ ]:
# Detect duplicates
print("Duplicate gene_ids:")
print(filled[filled.duplicated(subset='gene_id', keep=False)])

In [ ]:
# Strategy 1: Keep the first occurrence
deduped = filled.drop_duplicates(subset='gene_id', keep='first')
print(f"After drop_duplicates (keep='first'): {len(deduped)} rows")
print(deduped)

In [ ]:
# Strategy 2: Average the duplicates (common for technical replicates)
averaged = filled.groupby('gene_id', as_index=False).agg({
    'sample_1': 'mean',
    'sample_2': 'mean',
    'sample_3': 'mean',
    'gene_type': 'first',  # keep the first annotation
})
print(f"After averaging duplicates: {len(averaged)} rows")
print(averaged)

---

## Part 3: Type Conversion

Data loaded from CSV files often has columns stored as the wrong type (e.g., chromosome numbers as strings, p-values as text). Use `astype()` and `pd.to_numeric()` to fix this.

In [ ]:
# Messy clinical metadata
clinical = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'age': ['45', '52', '38', 'unknown', '61'],           # should be numeric
    'tumor_stage': ['II', 'III', 'I', 'II', 'IV'],
    'mutation_count': ['12', '8', '25', '3', 'N/A'],       # should be numeric
    'alive': ['True', 'True', 'False', 'True', 'False'],   # should be boolean
})

print("Original dtypes:")
print(clinical.dtypes)
print()
print(clinical)

In [ ]:
# Convert with error handling
clinical['age'] = pd.to_numeric(clinical['age'], errors='coerce')  # 'unknown' -> NaN
clinical['mutation_count'] = pd.to_numeric(clinical['mutation_count'], errors='coerce')
clinical['alive'] = clinical['alive'].map({'True': True, 'False': False})

print("Fixed dtypes:")
print(clinical.dtypes)
print()
print(clinical)

In [ ]:
# Categorical dtype -- saves memory and enables ordering
stage_order = ['I', 'II', 'III', 'IV']
clinical['tumor_stage'] = pd.Categorical(clinical['tumor_stage'],
                                          categories=stage_order,
                                          ordered=True)

print("Tumor stage is now categorical:")
print(clinical['tumor_stage'])
print(f"\nPatients with stage >= III:")
print(clinical[clinical['tumor_stage'] >= 'III'][['patient_id', 'tumor_stage']])

---

## Part 4: Reshaping Data -- Melt and Pivot

Biological data often needs to be converted between **wide** and **long** format.

```
WIDE (one row per gene):
  gene  | sample_1 | sample_2 | sample_3
  TP53  |   5.2    |   5.4    |   5.0

          melt()  <-->  pivot_table()

LONG (one row per measurement):
  gene  | sample   | expression
  TP53  | sample_1 |   5.2
  TP53  | sample_2 |   5.4
  TP53  | sample_3 |   5.0
```

- **Wide format** is natural for matrix computations (NumPy, heatmaps).
- **Long format** is required by most plotting libraries (seaborn, ggplot) and statistical models.

In [ ]:
# Start with a clean wide expression table
wide_expr = pd.DataFrame({
    'gene':     ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS'],
    'gene_type': ['TSG', 'TSG', 'Oncogene', 'Oncogene', 'Oncogene'],
    'ctrl_1':   [5.2, 3.8, 7.1, 2.4, 4.5],
    'ctrl_2':   [5.4, 3.6, 7.3, 2.6, 4.2],
    'treat_1':  [3.1, 5.9, 7.0, 4.8, 2.1],
    'treat_2':  [2.9, 6.2, 6.8, 5.1, 1.9],
})

print("Wide format:")
print(wide_expr)

In [ ]:
# Melt: wide -> long
long_expr = wide_expr.melt(
    id_vars=['gene', 'gene_type'],
    value_vars=['ctrl_1', 'ctrl_2', 'treat_1', 'treat_2'],
    var_name='sample',
    value_name='expression',
)

print("Long format (melted):")
print(long_expr)

In [ ]:
# Add a condition column derived from the sample name
long_expr['condition'] = long_expr['sample'].str.split('_').str[0]

print("With condition column:")
print(long_expr.head(8))

In [ ]:
# Pivot: long -> wide
wide_again = long_expr.pivot_table(
    index=['gene', 'gene_type'],
    columns='sample',
    values='expression',
).reset_index()

# Flatten the MultiIndex column names
wide_again.columns.name = None

print("Pivoted back to wide:")
print(wide_again)

### Stack and Unstack

`stack()` and `unstack()` work with hierarchical (MultiIndex) indices. They are useful when your data already has a MultiIndex.

In [ ]:
# Create a MultiIndex DataFrame
multi_df = wide_expr.set_index(['gene', 'gene_type'])[['ctrl_1', 'ctrl_2', 'treat_1', 'treat_2']]
print("MultiIndex DataFrame:")
print(multi_df)

# Stack: columns -> rows (wide -> long)
stacked = multi_df.stack()
stacked.name = 'expression'
print("\nStacked (long):")
print(stacked.head(8))

# Unstack: rows -> columns (long -> wide)
unstacked = stacked.unstack()
print("\nUnstacked (wide again):")
print(unstacked)

---

## Part 5: String Operations in Pandas

Gene annotation files often contain free-text columns that need parsing. Pandas provides vectorized string methods through the `.str` accessor.

In [ ]:
# Messy gene annotation table
annotations = pd.DataFrame({
    'raw_id': [
        'gene_id "ENSG00000141510"; gene_name "TP53"; gene_biotype "protein_coding";',
        'gene_id "ENSG00000012048"; gene_name "BRCA1"; gene_biotype "protein_coding";',
        'gene_id "ENSG00000146648"; gene_name "EGFR"; gene_biotype "protein_coding";',
        'gene_id "ENSG00000136997"; gene_name "MYC"; gene_biotype "protein_coding";',
        'gene_id "ENSG00000133703"; gene_name "KRAS"; gene_biotype "protein_coding";',
        'gene_id "ENSG00000228630"; gene_name "HOTAIR"; gene_biotype "lncRNA";',
    ]
})

print("Raw GTF-like attribute column:")
print(annotations)

In [ ]:
# Extract Ensembl ID using str.extract with regex
annotations['ensembl_id'] = annotations['raw_id'].str.extract(r'gene_id "(ENSG\d+)"')

# Extract gene name
annotations['gene_name'] = annotations['raw_id'].str.extract(r'gene_name "(\w+)"')

# Extract biotype
annotations['biotype'] = annotations['raw_id'].str.extract(r'gene_biotype "([^"]+)"')

print("Parsed annotations:")
print(annotations[['ensembl_id', 'gene_name', 'biotype']])

In [ ]:
# Common string operations
genes = pd.Series(['brca1', 'TP53', 'egfr', 'Myc', 'KRAS'])

print("Original:   ", genes.tolist())
print("Upper:      ", genes.str.upper().tolist())
print("Lower:      ", genes.str.lower().tolist())
print("Starts 'T': ", genes.str.upper().str.startswith('T').tolist())
print("Contains 'A':", genes.str.upper().str.contains('A').tolist())
print("Length:     ", genes.str.len().tolist())

In [ ]:
# Splitting strings: extract chromosome arm from cytogenetic band
bands = pd.Series(['17p13.1', '17q21.31', '7p11.2', '8q24.21', '12p12.1'])

# Split on the arm letter (p or q)
chromosomes = bands.str.extract(r'(\d+)[pq]')[0]
arms = bands.str.extract(r'\d+([pq])')[0]

band_df = pd.DataFrame({
    'band': bands,
    'chromosome': chromosomes,
    'arm': arms,
})
print(band_df)

---

## Part 6: Apply, Transform, and Pipe

These three methods let you apply custom functions to DataFrames.

| Method | Input | Returns | Use case |
|--------|-------|---------|----------|
| `apply()` | Row or column | Anything (scalar, Series) | General-purpose transformation |
| `transform()` | Row or column | Same-shape output | Group-wise normalization |
| `pipe()` | Entire DataFrame | DataFrame | Chaining processing steps |

### apply()

In [ ]:
# Apply a function to each row
expr_data = pd.DataFrame({
    'gene':     ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS'],
    'ctrl_1':   [5.2, 3.8, 7.1, 2.4, 4.5],
    'ctrl_2':   [5.4, 3.6, 7.3, 2.6, 4.2],
    'treat_1':  [3.1, 5.9, 7.0, 4.8, 2.1],
    'treat_2':  [2.9, 6.2, 6.8, 5.1, 1.9],
})

sample_cols = ['ctrl_1', 'ctrl_2', 'treat_1', 'treat_2']


def classify_expression(row):
    """Classify a gene based on its mean expression."""
    mean_val = row[sample_cols].mean()
    if mean_val > 6:
        return 'high'
    elif mean_val > 4:
        return 'medium'
    else:
        return 'low'


expr_data['category'] = expr_data.apply(classify_expression, axis=1)
print(expr_data[['gene', 'category']])

In [ ]:
# Apply a function column-wise: z-score normalize each sample
def zscore(col):
    return (col - col.mean()) / col.std()

normalized = expr_data[sample_cols].apply(zscore)
normalized.insert(0, 'gene', expr_data['gene'])

print("Z-score normalized:")
print(normalized.round(2))

### transform()

`transform()` is like `apply()` but guarantees the output has the same shape as the input. It is especially useful with `groupby()` for group-wise normalization.

In [ ]:
# Normalize expression within each gene_type group
long_data = wide_expr.melt(
    id_vars=['gene', 'gene_type'],
    value_vars=['ctrl_1', 'ctrl_2', 'treat_1', 'treat_2'],
    var_name='sample',
    value_name='expression',
)

# Group-wise z-score: normalize each gene_type group separately
long_data['zscore'] = long_data.groupby('gene_type')['expression'].transform(
    lambda x: (x - x.mean()) / x.std()
)

print("Group-wise z-scores:")
print(long_data.head(12).round(2))

### pipe()

`pipe()` lets you chain DataFrame-level functions into a readable pipeline.

In [ ]:
def remove_low_expression(df, threshold=3.0):
    """Remove genes where all samples are below threshold."""
    cols = [c for c in df.columns if c.startswith(('ctrl', 'treat'))]
    mask = df[cols].max(axis=1) >= threshold
    return df[mask]


def add_fold_change(df):
    """Add log2 fold change (treatment / control)."""
    ctrl_mean = df[['ctrl_1', 'ctrl_2']].mean(axis=1)
    treat_mean = df[['treat_1', 'treat_2']].mean(axis=1)
    df = df.copy()
    df['log2FC'] = np.log2(treat_mean / ctrl_mean)
    return df


def flag_significant(df, fc_threshold=0.5):
    """Flag genes with absolute fold change above threshold."""
    df = df.copy()
    df['significant'] = df['log2FC'].abs() > fc_threshold
    return df


# Chain the pipeline
result = (
    expr_data
    .pipe(remove_low_expression, threshold=2.5)
    .pipe(add_fold_change)
    .pipe(flag_significant, fc_threshold=0.5)
)

print("Pipeline result:")
print(result[['gene', 'log2FC', 'significant']].round(3))

---

## Part 7: Cleaning Messy Gene Annotation Tables

Real annotation files often have inconsistent naming, mixed case, extra whitespace, and embedded metadata. Here is a realistic cleaning workflow.

In [ ]:
# A messy annotation table (simulating what you might download)
messy_annot = pd.DataFrame({
    'gene_name':   [' TP53 ', 'brca1', 'EGFR', ' myc', 'KRAS', 'Tp53', 'pik3ca'],
    'chromosome':  ['chr17', 'chr17', 'Chr7', 'chr8', 'CHR12', 'chr17', 'chr3'],
    'start':       [7661779, 43044295, 55019021, 127735434, 25204789, 7661779, 179148114],
    'end':         [7687538, 43125483, 55211628, 127742951, 25250936, 7687538, 179240093],
    'description': [
        'tumor protein p53; cell cycle',
        'BRCA1 DNA repair; hereditary cancer',
        'epidermal growth factor receptor',
        'MYC proto-oncogene; bHLH transcription factor',
        'KRAS proto-oncogene, GTPase',
        'tumor protein p53; cell cycle',  # duplicate of TP53
        'PI3K catalytic subunit alpha',
    ],
})

print("Messy annotation table:")
print(messy_annot)

In [ ]:
# Step 1: Standardize gene names (strip whitespace, uppercase)
messy_annot['gene_name'] = messy_annot['gene_name'].str.strip().str.upper()

# Step 2: Standardize chromosome names (lowercase)
messy_annot['chromosome'] = messy_annot['chromosome'].str.lower()

# Step 3: Remove duplicates
clean_annot = messy_annot.drop_duplicates(subset='gene_name', keep='first')

# Step 4: Add computed columns
clean_annot = clean_annot.copy()
clean_annot['length'] = clean_annot['end'] - clean_annot['start'] + 1

# Step 5: Extract keywords from description
clean_annot['is_oncogene'] = clean_annot['description'].str.contains(
    'proto-oncogene|growth factor', case=False
)

print("Cleaned annotation table:")
print(clean_annot[['gene_name', 'chromosome', 'length', 'is_oncogene']])

---

## Part 8: Merging Gene Annotations with Expression Data

A common workflow: you have expression values keyed by Ensembl ID and annotations keyed by gene symbol. You need a bridge table.

In [ ]:
# Expression data (Ensembl IDs)
expression = pd.DataFrame({
    'ensembl_id': ['ENSG00000141510', 'ENSG00000012048', 'ENSG00000146648',
                   'ENSG00000136997', 'ENSG00000133703', 'ENSG00000228630'],
    'mean_expr': [5.2, 3.8, 7.1, 2.4, 4.5, 1.2],
})

# ID mapping table
id_map = pd.DataFrame({
    'ensembl_id': ['ENSG00000141510', 'ENSG00000012048', 'ENSG00000146648',
                   'ENSG00000136997', 'ENSG00000133703', 'ENSG00000228630'],
    'gene_name': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'HOTAIR'],
})

# Pathway annotations (gene symbols)
pathways = pd.DataFrame({
    'gene_name': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS'],
    'pathway': ['Apoptosis', 'DNA_Repair', 'EGFR_Signaling', 'Cell_Cycle', 'MAPK'],
})

# Two-step merge: expression -> id_map -> pathways
merged = (
    expression
    .merge(id_map, on='ensembl_id')
    .merge(pathways, on='gene_name', how='left')  # left join: keep HOTAIR even without pathway
)

print("Fully annotated expression:")
print(merged)

---

## Part 9: Handling Missing Values in Clinical Data

Clinical datasets have structured missingness: some patients drop out, some fields are optional, some values are censored.

In [ ]:
# Realistic clinical dataset
np.random.seed(42)
n = 20

clinical_data = pd.DataFrame({
    'patient_id': [f'PAT_{i:03d}' for i in range(n)],
    'age': np.random.choice([45, 52, 38, np.nan, 61, 55, 48, 67, np.nan, 44,
                             59, 71, 50, np.nan, 63, 42, 56, 49, 65, 53]),
    'sex': np.random.choice(['M', 'F'], n),
    'tumor_stage': np.random.choice(['I', 'II', 'III', 'IV', np.nan], n),
    'treatment': np.random.choice(['chemo', 'radiation', 'combo', np.nan], n),
    'survival_months': np.random.choice([12, 24, 36, 48, 60, np.nan, 18, 30], n),
})

print(f"Clinical dataset: {clinical_data.shape}")
print(f"\nMissing values per column:")
print(clinical_data.isna().sum())
print(f"\nPercent missing:")
print((clinical_data.isna().sum() / len(clinical_data) * 100).round(1))

In [ ]:
# Strategy: fill numeric columns with median, categorical with mode
cleaned_clinical = clinical_data.copy()

# Fill age with median
cleaned_clinical['age'] = cleaned_clinical['age'].fillna(
    cleaned_clinical['age'].median()
)

# Fill survival_months with median
cleaned_clinical['survival_months'] = cleaned_clinical['survival_months'].fillna(
    cleaned_clinical['survival_months'].median()
)

# Fill categorical with 'Unknown'
cleaned_clinical['tumor_stage'] = cleaned_clinical['tumor_stage'].fillna('Unknown')
cleaned_clinical['treatment'] = cleaned_clinical['treatment'].fillna('Unknown')

print("Missing values after cleaning:")
print(cleaned_clinical.isna().sum())
print("\nFirst 10 rows:")
print(cleaned_clinical.head(10))

In [ ]:
# Analyze: survival by treatment
survival_by_treatment = cleaned_clinical.groupby('treatment')['survival_months'].agg(
    ['count', 'mean', 'median']
).round(1)

print("Survival by treatment:")
print(survival_by_treatment)

---

## Part 10: Putting It All Together -- Expression Data Pipeline

Let us walk through a complete wrangling pipeline: start with messy wide-format expression data, clean it, reshape it, merge annotations, and produce a tidy result.

In [ ]:
# Step 1: Create messy input data
np.random.seed(42)

raw_expr = pd.DataFrame({
    'Gene_ID':  ['  tp53', 'BRCA1', 'egfr', 'MYC ', 'kras', 'Tp53', 'PIK3CA', 'PTEN'],
    'Sample_A1': [150.5, np.nan, 245.8, 312.4, 178.2, 148.9, 95.3, 63.1],
    'Sample_A2': [155.2, 89.3, 250.1, np.nan, 175.0, 152.1, 98.7, 60.5],
    'Sample_B1': [80.1, 180.5, 240.3, 620.8, 90.5, 78.3, 190.2, 120.4],
    'Sample_B2': [75.8, 175.2, np.nan, 610.1, 88.2, 82.5, 185.9, 115.8],
})

sample_metadata = pd.DataFrame({
    'sample': ['Sample_A1', 'Sample_A2', 'Sample_B1', 'Sample_B2'],
    'condition': ['control', 'control', 'treatment', 'treatment'],
    'batch': ['batch1', 'batch2', 'batch1', 'batch2'],
})

print("Raw expression data:")
print(raw_expr)
print("\nSample metadata:")
print(sample_metadata)

In [ ]:
# Step 2: Clean gene names
raw_expr['Gene_ID'] = raw_expr['Gene_ID'].str.strip().str.upper()
print("Gene names after cleaning:", raw_expr['Gene_ID'].tolist())

In [ ]:
# Step 3: Remove duplicates (average replicates)
expr_cols = ['Sample_A1', 'Sample_A2', 'Sample_B1', 'Sample_B2']
clean_expr = raw_expr.groupby('Gene_ID', as_index=False)[expr_cols].mean()
print(f"After deduplication: {len(clean_expr)} unique genes")
print(clean_expr)

In [ ]:
# Step 4: Fill missing values with gene (row) median
row_medians = clean_expr[expr_cols].median(axis=1)
for col in expr_cols:
    clean_expr[col] = clean_expr[col].fillna(row_medians)

print("After filling NaN:")
print(clean_expr.round(1))

In [ ]:
# Step 5: Reshape to long format
long = clean_expr.melt(
    id_vars='Gene_ID',
    value_vars=expr_cols,
    var_name='sample',
    value_name='expression',
)

# Step 6: Merge with sample metadata
long = long.merge(sample_metadata, on='sample')

print("Long format with metadata:")
print(long.head(12).round(1))

In [ ]:
# Step 7: Compute mean expression per gene per condition
summary = long.groupby(['Gene_ID', 'condition'])['expression'].mean().unstack()
summary['log2FC'] = np.log2(summary['treatment'] / summary['control'])
summary = summary.sort_values('log2FC')

print("Gene-level summary:")
print(summary.round(2))

---

## Summary

| Task | Key Methods |
|------|-------------|
| Missing values | `isna()`, `dropna()`, `fillna()` |
| Duplicates | `duplicated()`, `drop_duplicates()` |
| Type conversion | `astype()`, `pd.to_numeric()`, `pd.Categorical()` |
| Wide to long | `melt()`, `stack()` |
| Long to wide | `pivot_table()`, `unstack()` |
| String operations | `.str.upper()`, `.str.strip()`, `.str.extract()`, `.str.contains()` |
| Custom functions | `apply()`, `transform()`, `pipe()` |
| Combining tables | `merge()`, `concat()`, `join()` |

---

## Exercises

Work through the exercises below. Solutions are provided at the end of the notebook.

### Exercise 1: Clean and Impute

The following clinical dataset has missing values and inconsistencies. Clean it:

1. Standardize the `sex` column (should be 'M' or 'F').
2. Convert `age` to numeric (non-numeric values become NaN).
3. Fill missing `age` values with the median age.
4. Drop rows where `diagnosis` is missing.
5. Report the final shape and missing value count.

In [ ]:
messy_clinical = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007'],
    'age': ['45', '52', 'thirty-eight', '61', '55', '48', 'N/A'],
    'sex': ['Male', 'female', 'M', 'F', 'male', 'FEMALE', 'M'],
    'diagnosis': ['AML', 'CLL', np.nan, 'AML', 'ALL', 'CLL', 'AML'],
})

print(messy_clinical)

# Your code here


### Exercise 2: Reshape Expression Data

Given the wide expression matrix below:

1. Melt it to long format with columns: `gene`, `sample`, `expression`.
2. Add a `condition` column extracted from the sample name (everything before the underscore number).
3. Compute the mean expression per gene per condition.
4. Pivot back to wide format with conditions as columns.

In [ ]:
wide = pd.DataFrame({
    'gene': ['SOX2', 'NANOG', 'POU5F1', 'KLF4', 'MYC'],
    'stem_1': [8.5, 9.2, 7.8, 6.1, 3.4],
    'stem_2': [8.8, 9.0, 7.5, 6.3, 3.1],
    'diff_1': [2.1, 1.8, 2.5, 5.9, 7.2],
    'diff_2': [2.3, 2.0, 2.2, 6.0, 7.5],
})

print(wide)

# Your code here


### Exercise 3: Parse Annotation Strings

Extract structured information from these GTF-style attribute strings:

1. Extract the `gene_id` (ENSG...) into its own column.
2. Extract the `transcript_id` (ENST...) into its own column.
3. Extract the `gene_name` into its own column.
4. Count how many unique genes and transcripts are in the data.

In [ ]:
gtf_attrs = pd.DataFrame({
    'attribute': [
        'gene_id "ENSG00000141510"; transcript_id "ENST00000269305"; gene_name "TP53";',
        'gene_id "ENSG00000141510"; transcript_id "ENST00000420246"; gene_name "TP53";',
        'gene_id "ENSG00000012048"; transcript_id "ENST00000357654"; gene_name "BRCA1";',
        'gene_id "ENSG00000012048"; transcript_id "ENST00000471181"; gene_name "BRCA1";',
        'gene_id "ENSG00000146648"; transcript_id "ENST00000275493"; gene_name "EGFR";',
    ]
})

print(gtf_attrs)

# Your code here


### Exercise 4: Merge and Analyze

You have three tables: expression data, gene annotations, and GO terms. Perform:

1. Merge all three into a single table (use appropriate join types).
2. Find the GO category with the highest mean expression.
3. For each chromosome, count the number of genes and compute mean expression.

In [ ]:
expr_table = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN', 'RB1', 'APC'],
    'expression': [150, 89, 245, 312, 178, 63, 41, 95],
})

gene_annot = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN', 'RB1', 'APC'],
    'chromosome': ['chr17', 'chr17', 'chr7', 'chr8', 'chr12', 'chr10', 'chr13', 'chr5'],
})

go_terms = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN'],
    'go_category': ['Apoptosis', 'DNA_Repair', 'Signaling', 'Cell_Cycle',
                    'Signaling', 'Apoptosis'],
})

# Your code here


### Exercise 5: Complete Wrangling Pipeline

Build a pipeline using `pipe()` that:

1. Starts with the raw expression DataFrame below.
2. Cleans gene names (strip, uppercase).
3. Removes duplicate genes (keep the one with highest mean expression).
4. Fills missing values with gene-wise median.
5. Adds a `mean_expression` column.
6. Filters to genes with mean expression > 50.

Write each step as a separate function and chain with `pipe()`.

In [ ]:
raw = pd.DataFrame({
    'gene': ['  brca1', 'TP53', 'egfr ', 'MYC', ' kras', 'Tp53', 'PTEN'],
    's1': [89.3, 150.5, 245.8, 312.4, 178.2, 155.0, np.nan],
    's2': [np.nan, 148.9, 240.1, 310.0, 175.0, 152.1, 63.1],
    's3': [92.1, 155.2, np.nan, 315.8, 180.5, 148.5, 60.5],
})

print(raw)

# Your code here


### Exercise 6: Handle a Multi-Sheet Experiment

You receive data from a collaborator as two separate tables: one for expression values and one for quality metrics, linked by sample ID. Write code to:

1. Merge the two tables on `sample_id`.
2. Remove any samples with `rin_score` below 7.
3. Reshape the cleaned expression columns to long format.
4. Compute the mean expression per gene using only high-quality samples.

In [ ]:
# Expression data (wide: each row is a gene, columns are samples)
expr_wide = pd.DataFrame({
    'gene': ['SOX2', 'NANOG', 'POU5F1', 'KLF4'],
    'S001': [8.5, 9.2, 7.8, 6.1],
    'S002': [8.8, 9.0, 7.5, 6.3],
    'S003': [2.1, 1.8, 2.5, 5.9],
    'S004': [7.9, 8.5, 7.0, 5.8],
    'S005': [3.5, 3.0, 3.2, 5.5],
})

# Quality metrics (one row per sample)
quality = pd.DataFrame({
    'sample_id': ['S001', 'S002', 'S003', 'S004', 'S005'],
    'rin_score': [8.5, 6.2, 7.8, 8.1, 5.9],
    'total_reads_M': [25.3, 18.1, 22.7, 24.5, 15.0],
})

print("Expression data:")
print(expr_wide)
print("\nQuality metrics:")
print(quality)

# Your code here


---

## Solutions

### Solution 1

In [ ]:
messy_clinical = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007'],
    'age': ['45', '52', 'thirty-eight', '61', '55', '48', 'N/A'],
    'sex': ['Male', 'female', 'M', 'F', 'male', 'FEMALE', 'M'],
    'diagnosis': ['AML', 'CLL', np.nan, 'AML', 'ALL', 'CLL', 'AML'],
})

# 1. Standardize sex
messy_clinical['sex'] = messy_clinical['sex'].str.strip().str.upper().str[0]

# 2. Convert age to numeric
messy_clinical['age'] = pd.to_numeric(messy_clinical['age'], errors='coerce')

# 3. Fill missing age with median
messy_clinical['age'] = messy_clinical['age'].fillna(messy_clinical['age'].median())

# 4. Drop rows with missing diagnosis
messy_clinical = messy_clinical.dropna(subset=['diagnosis'])

# 5. Report
print(f"Final shape: {messy_clinical.shape}")
print(f"Missing values:\n{messy_clinical.isna().sum()}")
print("\nCleaned data:")
print(messy_clinical)

### Solution 2

In [ ]:
wide = pd.DataFrame({
    'gene': ['SOX2', 'NANOG', 'POU5F1', 'KLF4', 'MYC'],
    'stem_1': [8.5, 9.2, 7.8, 6.1, 3.4],
    'stem_2': [8.8, 9.0, 7.5, 6.3, 3.1],
    'diff_1': [2.1, 1.8, 2.5, 5.9, 7.2],
    'diff_2': [2.3, 2.0, 2.2, 6.0, 7.5],
})

# 1. Melt
long = wide.melt(
    id_vars='gene',
    value_vars=['stem_1', 'stem_2', 'diff_1', 'diff_2'],
    var_name='sample',
    value_name='expression',
)

# 2. Extract condition
long['condition'] = long['sample'].str.rsplit('_', n=1).str[0]

# 3. Mean per gene per condition
mean_expr = long.groupby(['gene', 'condition'])['expression'].mean().round(2)
print("Mean expression per gene per condition:")
print(mean_expr)

# 4. Pivot back to wide
wide_result = mean_expr.unstack()
print("\nPivoted:")
print(wide_result)

### Solution 3

In [ ]:
gtf_attrs = pd.DataFrame({
    'attribute': [
        'gene_id "ENSG00000141510"; transcript_id "ENST00000269305"; gene_name "TP53";',
        'gene_id "ENSG00000141510"; transcript_id "ENST00000420246"; gene_name "TP53";',
        'gene_id "ENSG00000012048"; transcript_id "ENST00000357654"; gene_name "BRCA1";',
        'gene_id "ENSG00000012048"; transcript_id "ENST00000471181"; gene_name "BRCA1";',
        'gene_id "ENSG00000146648"; transcript_id "ENST00000275493"; gene_name "EGFR";',
    ]
})

# 1. Extract gene_id
gtf_attrs['gene_id'] = gtf_attrs['attribute'].str.extract(r'gene_id "(ENSG\d+)"')

# 2. Extract transcript_id
gtf_attrs['transcript_id'] = gtf_attrs['attribute'].str.extract(r'transcript_id "(ENST\d+)"')

# 3. Extract gene_name
gtf_attrs['gene_name'] = gtf_attrs['attribute'].str.extract(r'gene_name "(\w+)"')

print("Parsed GTF attributes:")
print(gtf_attrs[['gene_id', 'transcript_id', 'gene_name']])

# 4. Count unique
print(f"\nUnique genes: {gtf_attrs['gene_id'].nunique()}")
print(f"Unique transcripts: {gtf_attrs['transcript_id'].nunique()}")

### Solution 4

In [ ]:
expr_table = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN', 'RB1', 'APC'],
    'expression': [150, 89, 245, 312, 178, 63, 41, 95],
})

gene_annot = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN', 'RB1', 'APC'],
    'chromosome': ['chr17', 'chr17', 'chr7', 'chr8', 'chr12', 'chr10', 'chr13', 'chr5'],
})

go_terms = pd.DataFrame({
    'gene': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS', 'PTEN'],
    'go_category': ['Apoptosis', 'DNA_Repair', 'Signaling', 'Cell_Cycle',
                    'Signaling', 'Apoptosis'],
})

# 1. Merge all three
merged = (
    expr_table
    .merge(gene_annot, on='gene')
    .merge(go_terms, on='gene', how='left')
)
print("Merged table:")
print(merged)

# 2. GO category with highest mean expression
go_expr = merged.groupby('go_category')['expression'].mean()
print(f"\nHighest mean expression GO category: {go_expr.idxmax()} ({go_expr.max():.1f})")

# 3. Per-chromosome summary
chr_summary = merged.groupby('chromosome').agg(
    n_genes=('gene', 'count'),
    mean_expression=('expression', 'mean'),
).round(1)
print("\nPer-chromosome summary:")
print(chr_summary)

### Solution 5

In [ ]:
raw = pd.DataFrame({
    'gene': ['  brca1', 'TP53', 'egfr ', 'MYC', ' kras', 'Tp53', 'PTEN'],
    's1': [89.3, 150.5, 245.8, 312.4, 178.2, 155.0, np.nan],
    's2': [np.nan, 148.9, 240.1, 310.0, 175.0, 152.1, 63.1],
    's3': [92.1, 155.2, np.nan, 315.8, 180.5, 148.5, 60.5],
})

sample_cols = ['s1', 's2', 's3']


def clean_names(df):
    df = df.copy()
    df['gene'] = df['gene'].str.strip().str.upper()
    return df


def remove_duplicates(df):
    df = df.copy()
    df['_mean'] = df[sample_cols].mean(axis=1)
    df = df.sort_values('_mean', ascending=False).drop_duplicates(subset='gene', keep='first')
    return df.drop(columns='_mean')


def fill_missing(df):
    df = df.copy()
    row_medians = df[sample_cols].median(axis=1)
    for col in sample_cols:
        df[col] = df[col].fillna(row_medians)
    return df


def add_mean(df):
    df = df.copy()
    df['mean_expression'] = df[sample_cols].mean(axis=1)
    return df


def filter_low(df, threshold=50):
    return df[df['mean_expression'] > threshold]


result = (
    raw
    .pipe(clean_names)
    .pipe(remove_duplicates)
    .pipe(fill_missing)
    .pipe(add_mean)
    .pipe(filter_low, threshold=50)
)

print("Pipeline result:")
print(result.round(1))

### Solution 6

In [ ]:
expr_wide = pd.DataFrame({
    'gene': ['SOX2', 'NANOG', 'POU5F1', 'KLF4'],
    'S001': [8.5, 9.2, 7.8, 6.1],
    'S002': [8.8, 9.0, 7.5, 6.3],
    'S003': [2.1, 1.8, 2.5, 5.9],
    'S004': [7.9, 8.5, 7.0, 5.8],
    'S005': [3.5, 3.0, 3.2, 5.5],
})

quality = pd.DataFrame({
    'sample_id': ['S001', 'S002', 'S003', 'S004', 'S005'],
    'rin_score': [8.5, 6.2, 7.8, 8.1, 5.9],
    'total_reads_M': [25.3, 18.1, 22.7, 24.5, 15.0],
})

# 1. Identify high-quality samples (RIN >= 7)
good_samples = quality[quality['rin_score'] >= 7]['sample_id'].tolist()
print(f"High-quality samples: {good_samples}")

# 2. Keep only those columns in expression data
expr_clean = expr_wide[['gene'] + good_samples]
print(f"\nFiltered expression matrix:")
print(expr_clean)

# 3. Reshape to long
expr_long = expr_clean.melt(
    id_vars='gene',
    var_name='sample_id',
    value_name='expression',
)

# 4. Merge with quality (optional, for context)
expr_long = expr_long.merge(quality, on='sample_id')

# 5. Mean per gene
mean_per_gene = expr_long.groupby('gene')['expression'].mean().round(2)
print(f"\nMean expression per gene (high-quality samples only):")
print(mean_per_gene)

---

**Previous module:** [Module 16 -- NumPy and Pandas](../16_NumPy_and_Pandas/01_numpy_and_pandas.ipynb)  
**Next module:** [Module 18 -- Data Visualization](../18_Data_Visualization/01_data_visualization.ipynb)

---[< Previous: NumPy and Pandas for Bioinformatics](../16_NumPy_and_Pandas/02_pandas.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Data Visualization for Bioinformatics >](../18_Data_Visualization/01_data_visualization.ipynb)---